# FREM-S12 Kaggle capability probe
This engineering-only probe records the current Kaggle runtime and attached-input topology. It does not install FREM-S12, read market data, fit a model, or admit scientific execution.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import socket
import sys
from pathlib import Path

import numpy as np

INPUT_ROOT = Path("/kaggle/input")
OUTPUT = Path("/kaggle/working/frem-s12-capability")
OUTPUT.mkdir(parents=True, exist_ok=True)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def egress() -> tuple[bool, str]:
    try:
        with socket.create_connection(("1.1.1.1", 443), timeout=2):
            return False, "direct_tcp_connection_succeeded"
    except OSError as exc:
        return True, f"direct_tcp_connection_failed:{type(exc).__name__}"

files = {}
for path in sorted(INPUT_ROOT.rglob("*")) if INPUT_ROOT.exists() else []:
    if path.is_file():
        files[str(path.relative_to(INPUT_ROOT))] = {
            "bytes": path.stat().st_size,
            "sha256": sha256(path),
        }

blocked, network_observation = egress()
input_flags = os.statvfs(INPUT_ROOT).f_flag if INPUT_ROOT.exists() else 0
working = shutil.disk_usage("/kaggle/working")
probe = {
    "schema_version": "frem-s12-kaggle-capability-v1",
    "engineering_only": True,
    "scientific_admitted": False,
    "python": platform.python_version(),
    "python_executable": sys.executable,
    "numpy": np.__version__,
    "numpy_path": str(Path(np.__file__).resolve()),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor_count": os.cpu_count(),
    "input_root_exists": INPUT_ROOT.is_dir(),
    "input_root_mode": oct(INPUT_ROOT.stat().st_mode) if INPUT_ROOT.exists() else None,
    "input_root_statvfs_read_only": bool(input_flags & getattr(os, "ST_RDONLY", 1)),
    "input_files": files,
    "network_disabled_observed": blocked,
    "network_observation": network_observation,
    "working_disk": {
        "total_bytes": working.total,
        "used_bytes": working.used,
        "free_bytes": working.free,
    },
}
path = OUTPUT / "capability.json"
path.write_text(json.dumps(probe, indent=2, sort_keys=True) + "\n")
print(json.dumps(probe, indent=2, sort_keys=True))
